# Dynamic Data Preparation - S5P, MODIS, CAMS

This notebook prepares satellite data with settings from **config.py**.

**To change settings:**
1. Edit `config.py` to change paths and CRS
2. Run this notebook

**No need to edit this notebook!**

In [19]:
# Import necessary libraries
from pathlib import Path
import gc

import osgeo
import xarray as xr
import geopandas as gpd
import numpy as np
from scipy import stats
from rasterio.enums import Resampling
import rioxarray as rxr
import odc.geo.xr

import warnings
warnings.filterwarnings("ignore")

In [20]:
# Load configuration
%run 00_config.ipynb
print_config()

CURRENT CONFIGURATION

Date Range:
  Start: 2024-01-01
  End: 2024-12-31

Processing Settings:
  Sample Period: 1YE
  Statistical Method: mean
  Region: All
  Pollutants: SO2, NO2, CO, O3, PM2P5, PM10

Output Path:
  C:\projects\aqi_scad\03_Output_Data\yearly_data


In [ ]:
# Define paths from config
download_path =PATHS['download']
converted_zarr_path = PATHS['input']

# Create lists of file paths
zarr_files = list(download_path.glob('*.zarr'))
cams_files = list(download_path.glob('*.grib'))

print(f"Found {len(zarr_files)} Zarr files")
print(f"Found {len(cams_files)} GRIB files")

Found 5 Zarr files
Found 1 GRIB files


In [22]:
# Load template for reprojection (using MODIS as highest resolution)
selected_zarr = list(filter(lambda x: 'MODIS' in str(x), zarr_files))[-1]
modis_da = xr.open_zarr(selected_zarr, chunks='auto')

# Create template image with target CRS from config
image = modis_da.isel(time=0).rename({'Y': 'y', 'X': 'x'}).transpose('y', 'x').rio.write_crs(CONFIG['target_crs'])
out_geobox = image.odc.geobox

print(f"Template geobox created with CRS: {CONFIG['target_crs']}")
print(f"Spatial resolution: {out_geobox.resolution}")

Template geobox created with CRS: EPSG:3857
Spatial resolution: Resolution(x=1000, y=1000)


In [23]:
# Unit conversion dictionaries
s5p_conv_dict = {
    'SO2': [64.066, 1000000, 1000],
    'NO2': [46.0055, 1000000, 1000],
    'CO': [28.01, 1000, 1000],
    'O3': [48, 10000, 1000]
}

cams_conv_dict = {
    'SO2': ['tcso2', 1000000000, 1000],
    'NO2': ['tcno2', 1000000000, 1000],
    'CO': ['tcco', 1000000, 1000],
    'O3': ['gtco3', 1000000, 1000],
    'PM2P5': ['pm2p5', 1000000000, 1],
    'PM10': ['pm10', 1000000000, 1]
}

In [24]:
def new_linregress(x, y):
    """
    Wrapper around scipy linregress to use in apply_ufunc.
    """
    mask = ~np.isnan(x) & ~np.isnan(y)
    x_m, y_m = x[mask], y[mask]
    try:
        slope, intercept, r_value, p_value, std_err = stats.linregress(x_m, y_m)
        return np.array([slope, intercept, r_value, p_value, std_err])
    except:
        return np.array([np.nan, np.nan, np.nan, np.nan, np.nan])

In [25]:
def convert_s5p(pollutant, zarr_list, output_path, out_geobox):
    """
    Converts S5P data from Zarr files to common geobox and unit.
    """
    da_list = []
    selected_zarr = list(filter(lambda x: f'S5P_{pollutant}' in str(x), zarr_list))
    
    for z in selected_zarr:
        da = xr.open_zarr(z, chunks='auto')
        
        if 'X' in list(da.coords):
            da = da.chunk({'time':2000, 'Y':len(da.Y), 'X':len(da.X)})
            da_list.append(da.rename({'Y': 'y', 'X': 'x'}).transpose('time','y', 'x').odc.assign_crs(CONFIG['target_crs']).odc.reproject(out_geobox, resampling="cubic_spline"))
        else:
            da = da.chunk({'time':2000, 'y':len(da.y), 'x':len(da.x)})
            da_list.append(da.transpose('time','y', 'x').odc.assign_crs(CONFIG['target_crs']).odc.reproject(out_geobox, resampling="cubic_spline"))
    
    da_all = xr.concat(da_list, dim='time')
    
    # Convert units
    conversion_factor = s5p_conv_dict[pollutant][0] * s5p_conv_dict[pollutant][1]
    da_vol = da_all / s5p_conv_dict[pollutant][2]
    da_conv = da_vol * conversion_factor
    
    # Save with appropriate unit suffix
    if pollutant == 'CO':
        da_conv.chunk('auto').to_zarr(output_path / f"S5P_{pollutant}_mgm3.zarr", mode='w')
    else:
        da_conv.chunk('auto').to_zarr(output_path / f"S5P_{pollutant}_ugm3.zarr", mode='w')
    
    del da_conv, da_vol, da, da_list, da_all
    gc.collect()

In [26]:
def convert_cams(pollutant, cams_list, output_path, out_geobox):
    """
    Converts CAMS data from GRIB files to common geobox and unit.
    """
    import shutil
    
    da_list = []
    
    for f in cams_list:
        ds = xr.load_dataset(f, engine="cfgrib", decode_coords="all")
        da = ds[cams_conv_dict[pollutant][0]]
        da_list.append(da)
    
    da_full = xr.concat(da_list, dim='time')
    da_full_conv = (da_full / cams_conv_dict[pollutant][2]) * cams_conv_dict[pollutant][1]
    
    # Reproject to target geobox
    da_full_conv_match = da_full_conv.rename({'latitude': 'x', 'longitude': 'y'}).transpose('time','y', 'x').odc.assign_crs("EPSG:4326").odc.reproject(out_geobox, resampling="cubic_spline")
    
    # Delete existing file before saving (Windows network drive fix)
    if pollutant == 'CO':
        output_file = output_path / f"CAMS_{pollutant}_mgm3.zarr"
    else:
        output_file = output_path / f"CAMS_{pollutant}_ugm3.zarr"
    
    if output_file.exists():
        shutil.rmtree(output_file)
    
    # Save with appropriate unit suffix
    da_full_conv_match.to_zarr(output_file, mode="w")
    
    del ds, da, da_list, da_full, da_full_conv, da_full_conv_match
    gc.collect()

In [27]:
def create_daily_mod_aod(zarr_list, output_path, overwrite=False):
    """
    Creates daily averaged MODIS AOD dataset from Zarr files.
    """
    selected_zarr = list(filter(lambda x: 'MODIS' in str(x), zarr_list))
    da_list = []
    
    for z in selected_zarr:
        print(f"  Loading: {z.name}")
        da = xr.open_zarr(z, chunks='auto')
        da_list.append(da)
    
    # Concatenate, sort, and resample to daily
    full_da = xr.concat(da_list, dim='time').sortby('time')
    full_da_1d = full_da.resample(time='1D').mean().rename({'Y': 'y', 'X': 'x'}).transpose('time','y', 'x')
    
    if overwrite:
        full_da_1d.chunk('auto').to_zarr(output_path / 'MOD_AOD550_1D.zarr', mode='w')
    else:
        full_da_1d.chunk('auto').to_zarr(output_path / 'MOD_AOD550_1D.zarr')

In [28]:
def aod_to_pollutant(pollutant, output_zarr_list, output_path, model_list=None, create_model=False):
    """
    Converts MODIS AOD to pollutant concentrations using linear regression.
    """
    # Open daily AOD
    aod_zarr = list(filter(lambda x: 'MOD_AOD550' in str(x), output_zarr_list))[0]
    aod_da = xr.open_zarr(aod_zarr, chunks='auto')
    
    if create_model:
        # Create linear regression model using CAMS data
        pol_zarr = list(filter(lambda x: f'CAMS_{pollutant}' in str(x), output_zarr_list))[0]
        cams_pol_da = xr.open_zarr(pol_zarr, chunks='auto').drop_vars(['number', 'step', 'surface', 'valid_time'])
        
        # Resample both to 2-week intervals
        aod_da_2w = aod_da.resample(time='2W').mean()
        cams_pol_da_2w = cams_pol_da.sortby('time').resample(time='2W').mean()
        
        data = cams_pol_da_2w.sel(time=aod_da_2w.time, method='nearest')
        data = data.drop_duplicates(dim='time')
        data['Optical_Depth_055'] = aod_da_2w.Optical_Depth_055
        data = data.compute()
        
        # Filter to IQR
        data_lqr = data.where(data.Optical_Depth_055 >= data.Optical_Depth_055.quantile(0.25), drop=True)
        data_lqr_uqr = data_lqr.where(data_lqr.Optical_Depth_055 <= data_lqr.Optical_Depth_055.quantile(0.75), drop=True)
        
        # Apply linear regression
        out_stats = xr.apply_ufunc(new_linregress, data_lqr_uqr['Optical_Depth_055'], data_lqr_uqr[cams_conv_dict[pollutant][0]],
                           input_core_dims=[['time'], ['time']],
                           output_core_dims=[["parameter"]],
                           vectorize=True,
                           dask="parallelized",
                           output_dtypes=['float64'],
                           dask_gufunc_kwargs={"output_sizes":{"parameter": 5}, "allow_rechunk":True}
                          )
        
        stats_da = out_stats.compute()
        slope = out_stats[:,:,0]
        intercept = out_stats[:,:,1]
        
        # Save regression model
        stats_da.transpose('parameter','y', 'x').odc.assign_crs(CONFIG['target_crs']).rio.to_raster(output_path / f"MOD-CAMS_{pollutant}_LRStats.tif")
    else:
        # Use existing linear regression model
        model_file = list(filter(lambda x: f'MOD-CAMS_{pollutant}_LRStats.tif' in str(x), model_list))[0]
        print(f"  Using model: {model_file.name}")
        
        model_da = rxr.open_rasterio(model_file, masked=True)
        slope = model_da.isel(band=0).to_numpy()
        intercept = model_da.isel(band=1).to_numpy()
    
    # Apply model to AOD
    aod_da = aod_da['Optical_Depth_055'].compute()
    mod_pol_1d_da = (aod_da * slope) + intercept
    
    # Save with appropriate unit suffix
    if pollutant == 'CO':
        mod_pol_1d_da.to_zarr(output_path / f"MOD_{pollutant}_mgm3.zarr")
    else:
        mod_pol_1d_da.to_zarr(output_path / f"MOD_{pollutant}_ugm3.zarr")

## Convert S5P Data

In [29]:
print("=" * 70)
print("CONVERTING S5P DATA")
print("=" * 70)

s5p_pollutants = ['SO2', 'NO2', 'CO', 'O3']

for pol in s5p_pollutants:
    print(f"\nConverting S5P {pol}...")
    convert_s5p(pol, zarr_files, converted_zarr_path, out_geobox)
    print(f"  ✓ {pol} conversion complete")

print("\n✓ S5P conversion complete!")

CONVERTING S5P DATA

Converting S5P SO2...
  ✓ SO2 conversion complete

Converting S5P NO2...
  ✓ NO2 conversion complete

Converting S5P CO...
  ✓ CO conversion complete

Converting S5P O3...
  ✓ O3 conversion complete

✓ S5P conversion complete!


## Convert CAMS Data

In [30]:
print("\n" + "=" * 70)
print("CONVERTING CAMS DATA")
print("=" * 70)

cams_pollutants = ['SO2', 'NO2', 'CO', 'O3', 'PM2P5', 'PM10']

for pol in cams_pollutants:
    print(f"\nConverting CAMS {pol}...")
    convert_cams(pol, cams_files, converted_zarr_path, out_geobox)
    print(f"  ✓ {pol} conversion complete")

print("\n✓ CAMS conversion complete!")

Ignoring index file 'C:\\projects\\aqi_scad\\01_Downloaded\\cams-global-reanalysis-eac4_2024-2025.grib.5b7b6.idx' incompatible with GRIB file



CONVERTING CAMS DATA

Converting CAMS SO2...


Ignoring index file 'C:\\projects\\aqi_scad\\01_Downloaded\\cams-global-reanalysis-eac4_2024-2025.grib.5b7b6.idx' incompatible with GRIB file


  ✓ SO2 conversion complete

Converting CAMS NO2...


Ignoring index file 'C:\\projects\\aqi_scad\\01_Downloaded\\cams-global-reanalysis-eac4_2024-2025.grib.5b7b6.idx' incompatible with GRIB file


...

  ✓ CO conversion complete

Converting CAMS O3...


Ignoring index file 'C:\\projects\\aqi_scad\\01_Downloaded\\cams-global-reanalysis-eac4_2024-2025.grib.5b7b6.idx' incompatible with GRIB file


  ✓ O3 conversion complete

Converting CAMS PM2P5...


Ignoring index file 'C:\\projects\\aqi_scad\\01_Downloaded\\cams-global-reanalysis-eac4_2024-2025.grib.5b7b6.idx' incompatible with GRIB file


  ✓ PM2P5 conversion complete

Converting CAMS PM10...
  ✓ PM10 conversion complete

✓ CAMS conversion complete!


## Convert MODIS AOD to Pollutants

In [32]:
print("\n" + "=" * 70)
print("CONVERTING MODIS AOD TO POLLUTANTS")
print("=" * 70)

# Create daily MODIS AOD
print("\nCreating daily MODIS AOD...")
create_daily_mod_aod(zarr_files, converted_zarr_path)
print("  ✓ Daily AOD created")

# Get list of output files and models
zarr_list_output = list(converted_zarr_path.glob('*.zarr'))
model_list_in = list(converted_zarr_path.glob('*.tif'))

modis_pollutants = ['SO2', 'NO2', 'CO', 'O3', 'PM2P5', 'PM10']

for pollutant in modis_pollutants:
    print(f"\nConverting MODIS {pollutant}...")
    aod_to_pollutant(pollutant, zarr_list_output, converted_zarr_path, model_list=model_list_in)
    print(f"  ✓ {pollutant} conversion complete")

print("\n✓ MODIS conversion complete!")


CONVERTING MODIS AOD TO POLLUTANTS

Creating daily MODIS AOD...
  Loading: MODIS_AOD055_2024-01-01_2024-12-31.zarr
  ✓ Daily AOD created

Converting MODIS SO2...
  Using model: MOD-CAMS_SO2_LRStats.tif
  ✓ SO2 conversion complete

Converting MODIS NO2...
  Using model: MOD-CAMS_NO2_LRStats.tif
  ✓ NO2 conversion complete

Converting MODIS CO...
  Using model: MOD-CAMS_CO_LRStats.tif
  ✓ CO conversion complete

Converting MODIS O3...
  Using model: MOD-CAMS_O3_LRStats.tif
  ✓ O3 conversion complete

Converting MODIS PM2P5...
  Using model: MOD-CAMS_PM2P5_LRStats.tif
  ✓ PM2P5 conversion complete

Converting MODIS PM10...
  Using model: MOD-CAMS_PM10_LRStats.tif
  ✓ PM10 conversion complete

✓ MODIS conversion complete!


## Validation

In [34]:
    print("\n" + "=" * 70)
    print("VALIDATION - Prepared Data")
    print("=" * 70)
    
    # Define all pollutants for each satellite
    satellite_pollutants = {
        'S5P': ['SO2', 'NO2', 'CO', 'O3'],
        'CAMS': ['SO2', 'NO2', 'CO', 'O3', 'PM2P5', 'PM10'],
        'MOD': ['SO2', 'NO2', 'CO', 'O3', 'PM2P5', 'PM10']
    }
    
    total_files = 0
    valid_files = 0
    errors = []
    
    for sat_name, pollutants in satellite_pollutants.items():
        print(f"\n{'='*70}")
        print(f"{sat_name} DATA VALIDATION")
        print(f"{'='*70}")
        
        for pollutant in pollutants:
            total_files += 1
            
            # Determine filename based on pollutant
            if pollutant == 'CO':
                filename = f'{sat_name}_{pollutant}_mgm3.zarr'
                unit = 'mg/m³'
            else:
                filename = f'{sat_name}_{pollutant}_ugm3.zarr'
                unit = 'µg/m³'
            
            zarr_file = converted_zarr_path / filename
            
            print(f"\n{pollutant}:")
            
            if zarr_file.exists():
                try:
                    ds = xr.open_zarr(zarr_file)
                    var_name = list(ds.keys())[0]
                    
                    data_min = float(ds[var_name].min().values)
                    data_max = float(ds[var_name].max().values)
                    nan_count = int(ds[var_name].isnull().sum().values)
                    total_count = ds[var_name].size
                    valid_count = total_count - nan_count
                    valid_pct = (valid_count / total_count) * 100
                    
                    print(f"  ✓ File: {filename}")
                    print(f"  Shape: {ds[var_name].shape} | Timesteps: {len(ds.time)}")
                    print(f"  Range: {data_min:.4f} to {data_max:.4f} {unit}")
                    print(f"  Valid: {valid_pct:.1f}% ({valid_count:,}/{total_count:,} pixels)")
                    
                    # Status check
                    if np.isnan(data_min) or np.isnan(data_max):
                        print(f"  ✗ ERROR: All data is NaN!")
                        errors.append(f"{sat_name} {pollutant}: All NaN")
                    elif data_min == data_max:
                        print(f"  ⚠️  WARNING: All values identical")
                        errors.append(f"{sat_name} {pollutant}: Constant value")
                    elif valid_pct < 1:
                        print(f"  ⚠️  WARNING: Very low coverage (<1%)")
                        errors.append(f"{sat_name} {pollutant}: Low coverage")
                    else:
                        print(f"  ✓ OK")
                        valid_files += 1
                        
                except Exception as e:
                    print(f"  ✗ ERROR reading file: {e}")
                    errors.append(f"{sat_name} {pollutant}: Read error")
            else:
                print(f"  ✗ File NOT found: {filename}")
                errors.append(f"{sat_name} {pollutant}: Missing file")
    
    print("\n" + "=" * 70)
    print("VALIDATION SUMMARY")
    print("=" * 70)
    print(f"Total files expected: {total_files}")
    print(f"Valid files: {valid_files}")
    print(f"Files with issues: {total_files - valid_files}")
    
    if errors:
        print(f"\n⚠️  Issues found:")
        for error in errors:
            print(f"  - {error}")
    else:
        print(f"\n✅ All files validated successfully!")
    
    print("\n" + "=" * 70)
    print("DATA PREPARATION COMPLETE")
    print("=" * 70)
    print(f"Output: {converted_zarr_path}")
    if valid_files == total_files:
        print(f"✅ Ready for normalization step!")
    else:
        print(f"⚠️  Some files have issues - check validation report above")


VALIDATION - Prepared Data

S5P DATA VALIDATION

SO2:
  ✓ File: S5P_SO2_ugm3.zarr
  Shape: (5154, 279, 503) | Timesteps: 5154
  Range: -64.0644 to 599.4913 µg/m³
  Valid: 6.8% (48,921,770/723,296,898 pixels)
  ✓ OK

NO2:
  ✓ File: S5P_NO2_ugm3.zarr
  Shape: (5150, 279, 503) | Timesteps: 5150
  Range: -1.1773 to 72.5977 µg/m³
  Valid: 7.5% (54,025,291/722,735,550 pixels)
  ✓ OK

CO:
  ✓ File: S5P_CO_mgm3.zarr
  Shape: (5155, 279, 503) | Timesteps: 5155
  Range: 0.2515 to 1.5005 mg/m³
  Valid: 4.3% (31,343,090/723,437,235 pixels)
  ✓ OK

O3:
  ✓ File: S5P_O3_ugm3.zarr
  Shape: (5159, 279, 503) | Timesteps: 5159
  Range: 50.0598 to 116.1536 µg/m³
  Valid: 7.5% (54,301,838/723,998,583 pixels)
  ✓ OK

CAMS DATA VALIDATION

SO2:
  ✓ File: CAMS_SO2_ugm3.zarr
  Shape: (2928, 279, 503) | Timesteps: 2928
  Range: 0.1227 to 6.1908 µg/m³
  Valid: 100.0% (410,906,736/410,906,736 pixels)
  ✓ OK

NO2:
  ✓ File: CAMS_NO2_ugm3.zarr
  Shape: (2928, 279, 503) | Timesteps: 2928
  Range: 0.7404 to 17.1647